# Phase 5.4 Preprocessing: Multi-Channel Retrieval Index Generation

This notebook extracts patient embeddings from a trained Run 31 baseline model, and then uses those embeddings to compute the similarity indices needed for the Phase 5.4 Multi-Channel Retrieval (MCR) configurations.

**Prerequisites:**
You MUST attach your trained baseline model (e.g., from Run 31 Phase 5.3) as a Kaggle Dataset, and update the path in Cell 3 (`CHECKPOINT_PATH`).

In [ ]:
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate pyarrow pyyaml datasets

In [ ]:
import os
import glob

if 'KAGGLE_URL_BASE' in os.environ:
    print('Running on Kaggle')
    os.chdir('/kaggle/working')
    os.system('rm -rf ./data ./src')
    os.makedirs('results', exist_ok=True)
    os.makedirs('data', exist_ok=True)
    
    train_paths = glob.glob('/kaggle/input/**/train.py', recursive=True)
    if train_paths:
        src_dir = os.path.dirname(train_paths[0])
        os.system(f'cp -r {src_dir} /kaggle/working/src')
        
    processed_paths = glob.glob('/kaggle/input/**/cohort_mimic3.pkl', recursive=True)
    if processed_paths:
        processed_dir = os.path.dirname(processed_paths[0])
        os.system(f'rm -rf ./data/processed && ln -s {processed_dir} ./data/processed')
        
    embeddings_paths = glob.glob('/kaggle/input/**/code_embeddings.pt', recursive=True)
    if embeddings_paths:
        embeddings_dir = os.path.dirname(embeddings_paths[0])
        os.system(f'rm -rf ./data/embeddings && ln -s {embeddings_dir} ./data/embeddings')

In [ ]:
# Set this to your pre-trained model checkpoint (e.g. from Phase 5.3)
CHECKPOINT_PATH = "/kaggle/input/my-baseline-model/best_model_seed42.pt"

!python src/extract_embeddings.py --model {CHECKPOINT_PATH} --output_dir data/embeddings --device cuda

In [ ]:
# 1. Standard EHR Similarity (Top 3)
!python src/compute_similarity.py --embeddings_dir data/embeddings --records data/processed/records_final.pkl --cohort data/processed/cohort_mimic3.pkl --output results/retrieval_index_top3.pkl --top_k 3 --note_sim_weight 0 --lab_sim_weight 0

In [ ]:
# 2. Notes-Only Similarity
!python src/compute_similarity.py --embeddings_dir data/embeddings --records data/processed/records_final.pkl --cohort data/processed/cohort_mimic3.pkl --output results/retrieval_index_notes_only.pkl --top_k 3 --note_embeddings data/processed/note_embeddings_mimic3.pkl --note_sim_weight 1.0 --lab_sim_weight 0

In [ ]:
# 3. Labs-Only Similarity
!python src/compute_similarity.py --embeddings_dir data/embeddings --records data/processed/records_final.pkl --cohort data/processed/cohort_mimic3.pkl --output results/retrieval_index_labs_jaccard.pkl --top_k 3 --note_sim_weight 0 --lab_sim_weight 1.0

In [ ]:
# 4. Trimodal Heuristic Similarity
!python src/compute_similarity.py --embeddings_dir data/embeddings --records data/processed/records_final.pkl --cohort data/processed/cohort_mimic3.pkl --output results/retrieval_index_trimodal_heuristic.pkl --top_k 3 --note_embeddings data/processed/note_embeddings_mimic3.pkl --note_sim_weight 0.33 --lab_sim_weight 0.33

### Done!
You can now download the 4 `.pkl` files from the `/kaggle/working/results` directory and upload them as a Kaggle Dataset to use in your Phase 5.4 training notebook!